# 5 — Canary Exposure Scoring

For each of the 24 trained models, compute the **canary exposure score** from
[Carlini et al. 2019 *The Secret Sharer*](https://arxiv.org/abs/1811.01062):

$$\text{Exposure}(s^*) = \log_2 |\mathcal{S}| - \log_2 \text{rank}(s^*)$$

where $|\mathcal{S}| = 900{,}000$ (all 6-digit secrets, 100000–999999) and
$\text{rank}(s^*)$ is the 1-indexed rank of the true secret by average
per-token log-probability (rank 1 = most likely → highest memorization).

Exposure = 0 means random guessing; exposure ≈ log₂(900000) ≈ 19.8 means
the model always ranks the true secret first.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/UofT/CSC2412/dp-finetuned-llms'
print('Drive mounted. Project dir:', PROJECT_DIR)

## 2. Install Dependencies

In [ ]:
!pip install -q transformers datasets matplotlib

## 3. Imports & Config

In [ ]:
import os
import gc
import json
import math
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from tqdm.auto import tqdm
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
from transformers.pytorch_utils import Conv1D

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

# ── Config ────────────────────────────────────────────────────────────────
CANARY_PREFIX = 'My patient ID is '
N_SECRETS     = 900_000            # all 6-digit integers: 100000–999999
SCORE_BATCH   = 256                # forward-pass batch size for scoring
MAX_SEQ_LEN   = 32                 # prefix (~8 tok) + secret (~3 tok) << 32

RESULTS_PATH   = f'{PROJECT_DIR}/results/dp_results.json'
CANARIES_PATH  = f'{PROJECT_DIR}/data/canaries.json'
EXPOSURE_PATH  = f'{PROJECT_DIR}/results/exposure_results.json'
PLOTS_DIR      = f'{PROJECT_DIR}/results/plots'
os.makedirs(PLOTS_DIR, exist_ok=True)

## 4. Load Canary Metadata & Training Results

In [ ]:
with open(CANARIES_PATH) as f:
    canaries = json.load(f)

with open(RESULTS_PATH) as f:
    dp_results = json.load(f)

# Build freq → canary lookup
freq_to_canary = {c['freq']: c for c in canaries}

print('Canaries:')
for c in canaries:
    print(f"  C{c['id']}  freq={c['freq']:>2}x  secret={c['secret']}  text=\"{c['text']}\"")

print(f'\nDP results: {len(dp_results)} runs')
print(f'\nFreq → canary map: {list(freq_to_canary.keys())}')

## 5. Build Candidate List

All 900 000 possible 6-digit secrets (100 000–999 999).

In [ ]:
all_secrets = [str(n) for n in range(100_000, 1_000_000)]
print(f'Candidate pool size: {len(all_secrets):,}  (e.g. {all_secrets[0]}, ..., {all_secrets[-1]})')

## 6. Scoring Helper

Compute the average per-token log-probability of `prefix + secret` for every
candidate in batches.  Ranking by this score is equivalent to ranking by
$\log P(\text{secret} \mid \text{prefix})$ because the prefix probability is
identical across all candidates.

In [ ]:
# ── Architecture helpers (mirrors 4_dp_train.ipynb) ──────────────────────
def replace_conv1d_with_linear(model):
    """Replace HuggingFace Conv1D with nn.Linear (Opacus training used this)."""
    for name, module in list(model.named_children()):
        if isinstance(module, Conv1D):
            in_f, out_f = module.weight.shape
            linear = torch.nn.Linear(in_f, out_f, bias=module.bias is not None)
            linear.weight = torch.nn.Parameter(module.weight.T.detach().clone())
            if module.bias is not None:
                linear.bias = torch.nn.Parameter(module.bias.detach().clone())
            setattr(model, name, linear)
        else:
            replace_conv1d_with_linear(module)
    return model


def load_checkpoint(model_path, tokenizer, device=device):
    """Load a checkpoint saved by 4_dp_train.ipynb.

    The checkpoints were saved from models whose Conv1D layers had been
    replaced with nn.Linear, so we must rebuild that architecture before
    loading the state dict — otherwise shape mismatches occur.
    """
    # Build modified architecture
    model = GPT2LMHeadModel.from_pretrained('gpt2')
    model.resize_token_embeddings(len(tokenizer))
    model = replace_conv1d_with_linear(model)
    model.lm_head.weight = torch.nn.Parameter(
        model.lm_head.weight.detach().clone())

    # Load saved weights (save_pretrained writes safetensors or .bin)
    safetensors_path = os.path.join(model_path, 'model.safetensors')
    bin_path         = os.path.join(model_path, 'pytorch_model.bin')
    if os.path.exists(safetensors_path):
        from safetensors.torch import load_file
        state_dict = load_file(safetensors_path, device='cpu')
    else:
        state_dict = torch.load(bin_path, map_location='cpu')
    model.load_state_dict(state_dict, strict=True)

    return model.half().to(device).eval()


# ── Scoring ───────────────────────────────────────────────────────────────
def score_candidates(model, tokenizer, prefix, candidates,
                     batch_size=SCORE_BATCH, device=device):
    """Return avg per-token log-prob of (prefix + candidate) for each candidate."""
    model.eval()
    all_scores = []

    with torch.no_grad():
        for i in tqdm(range(0, len(candidates), batch_size),
                      desc='scoring', leave=False):
            batch = candidates[i : i + batch_size]
            full_texts = [prefix + c for c in batch]

            enc = tokenizer(
                full_texts,
                return_tensors='pt',
                padding=True,
                truncation=True,
                max_length=MAX_SEQ_LEN,
            )
            input_ids = enc['input_ids'].to(device)       # (B, L)
            attn_mask = enc['attention_mask'].to(device)  # (B, L)

            outputs = model(input_ids=input_ids, attention_mask=attn_mask)
            logits  = outputs.logits  # (B, L, V)

            # Shift: logit[t] predicts token[t+1]
            shift_logits = logits[:, :-1, :].float()  # (B, L-1, V)
            shift_labels = input_ids[:, 1:]            # (B, L-1)

            tok_lp = torch.log_softmax(shift_logits, dim=-1)\
                           .gather(2, shift_labels.unsqueeze(-1))\
                           .squeeze(-1)  # (B, L-1)

            # Mask padding
            pad_mask = (shift_labels != tokenizer.pad_token_id).float()
            n_tok    = pad_mask.sum(dim=-1).clamp(min=1)
            score    = (tok_lp * pad_mask).sum(dim=-1) / n_tok  # (B,)

            all_scores.extend(score.cpu().tolist())

    return np.array(all_scores)


def compute_exposure(scores, true_secret, all_secrets):
    """Compute rank and exposure of the true secret."""
    true_idx   = all_secrets.index(true_secret)
    true_score = scores[true_idx]
    rank = int((scores > true_score).sum()) + 1
    exposure = math.log2(len(all_secrets)) - math.log2(rank)
    return rank, exposure, true_score


print('Helpers defined.')

## 7. Compute Exposure Scores

Load each of the 24 checkpoints, score all 900 000 candidates, and record
rank + exposure.  Completed runs are skipped — safe to interrupt and re-run.

In [ ]:
# Resume support
if os.path.exists(EXPOSURE_PATH):
    with open(EXPOSURE_PATH) as f:
        exposure_results = json.load(f)
    done_keys = {(r['run']) for r in exposure_results}
    print(f'Loaded {len(exposure_results)} existing exposure results.')
else:
    exposure_results = []
    done_keys = set()
    print('Starting fresh.')

tokenizer = GPT2TokenizerFast.from_pretrained(
    dp_results[0]['model_path'])  # any checkpoint has the tokenizer

for r in sorted(dp_results, key=lambda x: (x['epsilon_target'] or 999, x['freq'])):
    run_name = r['run']
    if run_name in done_keys:
        print(f'[skip] {run_name}')
        continue

    freq   = r['freq']
    canary = freq_to_canary[freq]
    eps_t  = r['epsilon_target']   # None for no-DP
    eps_label = 'inf' if eps_t is None else str(eps_t)

    print(f'\n[{run_name}]  ε={eps_label}  freq={freq}x  '
          f'canary secret={canary["secret"]}')

    # Load model — must rebuild modified architecture first (Conv1D → nn.Linear)
    model = load_checkpoint(r['model_path'], tokenizer)

    # Score all 900k candidates
    scores = score_candidates(model, tokenizer, CANARY_PREFIX, all_secrets)

    rank, exposure, true_score = compute_exposure(
        scores, canary['secret'], all_secrets)

    print(f'  rank={rank:,}  exposure={exposure:.3f}  '
          f'true_score={true_score:.4f}')

    exposure_results.append({
        'run':              run_name,
        'epsilon_target':   eps_t,
        'epsilon_achieved': r['epsilon_achieved'],
        'freq':             freq,
        'canary_secret':    canary['secret'],
        'rank':             rank,
        'exposure':         exposure,
        'true_log_prob':    float(true_score),
        'perplexity':       r['perplexity'],
    })

    # Save after each run
    with open(EXPOSURE_PATH, 'w') as f:
        json.dump(exposure_results, f, indent=2)

    del model
    torch.cuda.empty_cache()
    gc.collect()

print('\nAll exposure scores computed.')

## 8. Results Summary

In [ ]:
with open(EXPOSURE_PATH) as f:
    exposure_results = json.load(f)

print(f'=== Canary Exposure Results ({len(exposure_results)} runs) ===')
print(f'{"Run":<22}  {"ε":>6}  {"freq":>5}x  '
      f'{"rank":>10}  {"exposure":>9}  {"perplexity":>11}')
print('-' * 75)
for r in sorted(exposure_results,
                key=lambda x: (x['epsilon_target'] or 999, x['freq'])):
    eps = 'inf' if r['epsilon_target'] is None else f"{r['epsilon_target']:.1f}"
    print(f"{r['run']:<22}  {eps:>6}  {r['freq']:>5}x  "
          f"{r['rank']:>10,}  {r['exposure']:>9.3f}  {r['perplexity']:>11.2f}")

## 9. Plots

In [ ]:
import matplotlib
matplotlib.rcParams.update({
    'figure.dpi': 130,
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

FREQS   = [1, 5, 10, 50]
# x-axis: ε values in order (use a large finite number for ∞)
EPS_MAP = {0.5: 0.5, 1.0: 1.0, 2.0: 2.0, 4.0: 4.0, 8.0: 8.0, None: 16.0}
EPS_TICKS      = [0.5, 1, 2, 4, 8, 16]
EPS_TICK_LABELS = ['0.5', '1', '2', '4', '8', '∞']
COLORS  = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
MARKERS = ['o', 's', '^', 'D']

def gather(results, freq):
    rows = [r for r in results if r['freq'] == freq]
    rows.sort(key=lambda r: EPS_MAP[r['epsilon_target']])
    xs   = [EPS_MAP[r['epsilon_target']] for r in rows]
    exps = [r['exposure']   for r in rows]
    ppls = [r['perplexity'] for r in rows]
    return xs, exps, ppls

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))

for freq, color, marker in zip(FREQS, COLORS, MARKERS):
    xs, exps, _ = gather(exposure_results, freq)
    ax.plot(xs, exps, marker=marker, color=color, linewidth=1.8,
            markersize=7, label=f'freq={freq}×')

# Random-guess baseline
ax.axhline(0, color='grey', linewidth=1, linestyle='--', label='random (exp=0)')

ax.set_xscale('log')
ax.set_xticks(EPS_TICKS)
ax.set_xticklabels(EPS_TICK_LABELS)
ax.xaxis.set_minor_locator(ticker.NullLocator())
ax.set_xlabel('Privacy budget ε  (log scale)', labelpad=6)
ax.set_ylabel('Canary exposure  (bits)', labelpad=6)
ax.set_title('Canary Exposure vs Privacy Budget', pad=10)
ax.legend(frameon=False)

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/exposure_vs_epsilon.png', bbox_inches='tight')
plt.show()
print('Saved exposure_vs_epsilon.png')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))

for freq, color, marker in zip(FREQS, COLORS, MARKERS):
    xs, _, ppls = gather(exposure_results, freq)
    ax.plot(xs, ppls, marker=marker, color=color, linewidth=1.8,
            markersize=7, label=f'freq={freq}×')

# Baseline perplexity (no DP, no canaries) from 2_train.ipynb
BASELINE_PPL = 23.76
ax.axhline(BASELINE_PPL, color='grey', linewidth=1, linestyle='--',
           label=f'baseline (no canaries) = {BASELINE_PPL}')

ax.set_xscale('log')
ax.set_xticks(EPS_TICKS)
ax.set_xticklabels(EPS_TICK_LABELS)
ax.xaxis.set_minor_locator(ticker.NullLocator())
ax.set_xlabel('Privacy budget ε  (log scale)', labelpad=6)
ax.set_ylabel('Perplexity', labelpad=6)
ax.set_title('Validation Perplexity vs Privacy Budget', pad=10)
ax.legend(frameon=False)

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/perplexity_vs_epsilon.png', bbox_inches='tight')
plt.show()
print('Saved perplexity_vs_epsilon.png')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

for freq, color, marker in zip(FREQS, COLORS, MARKERS):
    xs, exps, ppls = gather(exposure_results, freq)
    ax1.plot(xs, exps, marker=marker, color=color, linewidth=1.8,
             markersize=7, label=f'freq={freq}×')
    ax2.plot(xs, ppls, marker=marker, color=color, linewidth=1.8,
             markersize=7)

ax1.axhline(0, color='grey', linewidth=1, linestyle='--', label='random')
ax2.axhline(BASELINE_PPL, color='grey', linewidth=1, linestyle='--',
            label=f'baseline {BASELINE_PPL}')

for ax in (ax1, ax2):
    ax.set_xscale('log')
    ax.set_xticks(EPS_TICKS)
    ax.set_xticklabels(EPS_TICK_LABELS)
    ax.xaxis.set_minor_locator(ticker.NullLocator())
    ax.set_xlabel('Privacy budget ε', labelpad=6)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

ax1.set_ylabel('Canary exposure (bits)', labelpad=6)
ax1.set_title('Memorization', pad=8)
ax1.legend(frameon=False, fontsize=9)

ax2.set_ylabel('Perplexity', labelpad=6)
ax2.set_title('Utility', pad=8)
ax2.legend(frameon=False, fontsize=9)

fig.suptitle('Privacy–Utility Trade-off  (DP-SGD on GPT-2)', y=1.02,
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/privacy_utility_tradeoff.png',
            bbox_inches='tight')
plt.show()
print('Saved privacy_utility_tradeoff.png')